In [21]:
import pandas as pd
import ast
from sklearn.feature_extraction.text import CountVectorizer
from nltk.stem.porter import PorterStemmer
from sklearn.metrics.pairwise import cosine_similarity

# loading data file
movies = pd.read_csv('F:\DataScience\Datasets\TMDB_movie dataset/tmdb_5000_movies.csv') 
credits = pd.read_csv('F:\DataScience\Datasets\TMDB_movie dataset/tmdb_5000_credits.csv')

# merging 2 files
movies = movies.merge(credits,on='title')

# taking required columns from the dataset
movies = movies[['movie_id','title','overview','genres','keywords','cast','crew']]

# Pre - Processing of data
    # checking the null values
print('number of null values in each column:\n',movies.isnull().sum())

# when compared with 5000 columns there are only 3 rows of null values in the overview column so, we can remove them
    # removing null value rows
movies.dropna(inplace=True)

# checking for duplicated values
print('\ncount of duplicated values: ',movies.duplicated().sum())

# some columns contain list of dictionaries we need to bring them into list of elements.
# the the list in the column  genres, keywords,cast and crew are objects (that means strings) we need to convert them into list
# to convert string to list we need to import ast.
def convert(str):
    lists = ast.literal_eval(str)
    l =[]
    for i in lists:         # here we are extracting each dictionary from the list
        l.append(i['name']) # {"id": 28, "name": "Action"} this is the dict format we need to extract only name value    
    return l

movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)

# i require only the top 3 actors names from the crew since most of them are interested in the hero and heroin
# so, we will define another function that extract only top 3 names from the cast column
def top3(str):
    lists = ast.literal_eval(str)
    l = []
    counter = 0
    for i in lists:
        if counter != 3:
            l.append(i['name']) # {"cast_id": 242, "character": "Jake Sully", "credit_id": "5602a8a7c3a3685532001c9a", "gender": 2, "id": 65731, "name": "Sam Worthington", "order": 0}
            counter += 1        # each dict contains character and name character is the name of the lead in the movie and name is the real life name like in movie "Billa" character name is "Billa"
                                # but his real name is 'Prabhas'.
        else:
            break
    return l
movies['cast'] = movies['cast'].apply(top3)

# from the crew dict we need only the director so we create other function for it
# {"credit_id": "52fe48009251416c750aca23", "department": "Editing", "gender": 0, "id": 1721, "job": "Editor", "name": "Stephen E. Rivkin"}
# as you see a specific key names 'job' which mentions the role of the person so to extract director we need key as 'job' to get the name of the director
def director(str):
    lists = ast.literal_eval(str)
    l=[]
    for i in lists:
        if i['job'].lower() == 'director':
            l.append(i['name'])
    return l
movies['crew'] = movies['crew'].apply(director)

# so we are ready with the clean and proper dataset but there is still one problem i.e the space between the names 
# for example the name James Cameron which has space between word 'James' and 'Cameron' so when we try to merge the columns
# cast crew overview  and keywords they get split and will be understood as 2 different words so if there is another person
# with name starting or ending with 'James' the model gets confused wether it need to call for 'James Cameron' or the person 'James'
# so to address that problem we are removing the spaces between words.
movies['genres'] = movies['genres'].apply(lambda x: [i.replace(' ','') for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x: [i.replace(' ','') for i in x])  
movies[['cast','crew']] = movies[['cast','crew']].applymap(lambda x: [i.replace(' ','') for i in x])


# Merging overview, genres, keywords, cast and crew
# as we are merging every column is in list form except the overview so, we need to convert that into list using split function
movies['overview'] = [i.split() for i in movies['overview']]
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

# creating new data frame
new_df = movies[['movie_id','title','tags']]

# creating a single string with lower case for the tags column
new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x).lower())

# changing the tags into vectors such they can be compared
cv = CountVectorizer(max_features=5000,stop_words='english')
vectorizer = cv.fit_transform(new_df['tags']).toarray()     # CountVectorizer().fit_transform gives the sparse matrix so we are converting it into numpy array using .toarray()

# using stemming to avoid repeated words. for example: acting and act, hero and heroes, etc.
def stem(text):
    ps = PorterStemmer()    
    list = []
    for i in text.split():
        list.append(ps.stem(i))
    return " ".join(list)
new_df['tags'] = new_df['tags'].apply(stem)

# to measure the similarity between any movies we are using cosine similarity
# we can also use euclidean distance but it doesn't work well for larger dimensional data so we use 
# cosine distance(measures the angle between 2 movies less "angle" more "similarity") 
# or cosine similarity(it's value is in between 0 and 1 where 0 represents "no similarity" and 1 represents "perfect similarity")
similarity = cosine_similarity(vectorizer)

# creating function to extract top 5 similar movies 
def recommend(movie):
    movie_index = new_df[new_df['title'] == movie].index[0]
    distances = similarity[movie_index]
    sorted_distances = sorted(list(enumerate(distances)),reverse=True,key = lambda x:x[1])  # sorting values of similarities in descending order based on similarity values
    for i in sorted_distances[1:6]:
        print(new_df.iloc[i[0]].title)

       


number of null values in each column:
 movie_id    0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64

count of duplicated values:  0


C:\Users\Admin\AppData\Local\Temp\ipykernel_14804\4022293183.py:76: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  movies[['cast','crew']] = movies[['cast','crew']].applymap(lambda x: [i.replace(' ','') for i in x])
C:\Users\Admin\AppData\Local\Temp\ipykernel_14804\4022293183.py:88: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x).lower())
C:\Users\Admin\AppData\Local\Temp\ipykernel_14804\4022293183.py:101: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.

In [22]:
recommend('Pirates of the Caribbean: At World\'s End')

Pirates of the Caribbean: Dead Man's Chest
Pirates of the Caribbean: On Stranger Tides
Pirates of the Caribbean: The Curse of the Black Pearl
20,000 Leagues Under the Sea
Puss in Boots


In [18]:
new_df

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a parapleg marin is dispa..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believ to be dead, ha c..."
2,206647,Spectre,a cryptic messag from bond’ past send him on a...
3,49026,The Dark Knight Rises,follow the death of district attorney harvey d...
4,49529,John Carter,"john carter is a war-weary, former militari ca..."
...,...,...,...
4804,9367,El Mariachi,el mariachi just want to play hi guitar and ca...
4805,72766,Newlyweds,a newlyw couple' honeymoon is upend by the arr...
4806,231617,"Signed, Sealed, Delivered","""signed, sealed, delivered"" introduc a dedic q..."
4807,126186,Shanghai Calling,when ambiti new york attorney sam is sent to s...
